# Week 17: Production Retrieval-Augmented Generation (Advanced RAG) & Vector Search
## Notebook 02: Hybrid Search (BM25 + Dense Vectors) & Reciprocal Rank Fusion (RRF)

### 1. Conceptual Intuition & Real-World Use Case
In production RAG systems, relying exclusively on dense vector embeddings leads to critical retrieval failures:
- **Dense Embedding Weakness:** Semantic models struggle with exact keyword lookups: model numbers (`X-4029B`), error codes (`ERR_SSL_PROTOCOL_ERROR`), drug names, or specific employee IDs.
- **Sparse BM25 Weakness:** Lexical search fails on conceptual synonyms (e.g. searching "automobile repair" misses documents containing "car maintenance").

**Hybrid Search** combines the best of both worlds:
1. **BM25 Sparse Lexical Search:** Catches exact keywords, acronyms, and codes.
2. **Dense Vector Search:** Catches broad semantic context and conceptual paraphrasing.
3. **Reciprocal Rank Fusion (RRF):** Fuses ranking lists without needing score normalization.

---

### 2. Mathematical Foundations
1. **Okapi BM25:**
   $$\text{Score}(D, Q) = \sum_{q_i \in Q} \text{IDF}(q_i) \times \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \left( 1 - b + b \cdot \frac{|D|}{\text{avgdl}} \right)}$$
   - $k_1 \approx 1.2 - 2.0$: Term frequency saturation parameter.
   - $b \approx 0.75$: Document length penalty parameter.

2. **Reciprocal Rank Fusion (RRF):**
   Combining raw scores directly is problematic because BM25 produces unbounded positive numbers ($[0, 50+]$) while cosine similarity is bounded in $[-1, 1]$.
   RRF bypasses score calibration by fusing **ranks** directly:
   $$\text{RRF\_Score}(d \in D) = \sum_{m \in M} \frac{1}{k + r_m(d)}$$
   Where:
   - $M$ is the set of retrieval models (BM25 and Dense).
   - $r_m(d)$ is the 1-based rank of document $d$ in system $m$.
   - $k \approx 60$ is a smoothing constant preventing top-ranked outliers from dominating.

In [ ]:
import math
from collections import Counter
import numpy as np

# Step 1: Corpus of Technical Documentation
documents = [
    "Error ERR_404: The requested database cluster could not be found.",
    "System memory usage exceeded 95 percent threshold on primary node.",
    "Guide to troubleshooting network connection issues and latency spikes.",
    "Error ERR_404: Missing authentication token in request header.",
    "Database indexing strategies for high-throughput Postgres read queries."
]

query = "how to fix ERR_404 database error"

# Step 2: Implement BM25 from Scratch
def tokenize(text):
    return text.lower().replace(":", "").replace(".", "").split()

tokenized_docs = [tokenize(doc) for doc in documents]
query_tokens = tokenize(query)
N = len(documents)
avgdl = np.mean([len(d) for d in tokenized_docs])

# Calculate Inverse Document Frequency (IDF)
idf = {}
for term in set(query_tokens):
    doc_freq = sum(1 for d in tokenized_docs if term in d)
    # BM25 probabilistic IDF formula
    idf[term] = math.log((N - doc_freq + 0.5) / (doc_freq + 0.5) + 1.0)

# Compute BM25 scores
k1 = 1.5
b = 0.75
bm25_scores = []
for doc in tokenized_docs:
    score = 0.0
    doc_len = len(doc)
    counts = Counter(doc)
    for term in query_tokens:
        if term in counts:
            freq = counts[term]
            num = freq * (k1 + 1)
            denom = freq + k1 * (1 - b + b * (doc_len / avgdl))
            score += idf[term] * (num / denom)
    bm25_scores.append(score)

bm25_ranking = np.argsort(bm25_scores)[::-1]
print("--- BM25 Rankings (Sparse Lexical) ---")
for rank, idx in enumerate(bm25_ranking, 1):
    print(f"  Rank {rank}: [Doc {idx}] (Score: {bm25_scores[idx]:.3f}) -> {documents[idx][:55]}...")

# Step 3: Simulate Dense Vector Search (Cosine Similarity)
# Suppose dense model scores general semantic relatedness
np.random.seed(99)
dense_scores = np.array([0.72, 0.40, 0.81, 0.65, 0.78])
dense_ranking = np.argsort(dense_scores)[::-1]

print("\n--- Dense Embeddings Rankings (Semantic) ---")
for rank, idx in enumerate(dense_ranking, 1):
    print(f"  Rank {rank}: [Doc {idx}] (Score: {dense_scores[idx]:.3f}) -> {documents[idx][:55]}...")

# Step 4: Reciprocal Rank Fusion (RRF) with k=60
k = 60
rrf_scores = {i: 0.0 for i in range(N)}

# Add BM25 rank scores
for rank_0, doc_id in enumerate(bm25_ranking):
    rank_1 = rank_0 + 1
    rrf_scores[doc_id] += 1.0 / (k + rank_1)

# Add Dense rank scores
for rank_0, doc_id in enumerate(dense_ranking):
    rank_1 = rank_0 + 1
    rrf_scores[doc_id] += 1.0 / (k + rank_1)

hybrid_ranking = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)

print("\n--- Final Hybrid Search (RRF Combined) ---")
for rank, idx in enumerate(hybrid_ranking, 1):
    print(f"  Rank {rank}: [Doc {idx}] (RRF Score: {rrf_scores[idx]:.5f}) -> {documents[idx]}")

### 3. Step-by-Step Code Walkthrough
1. **BM25 Precision:** Document 0 and Document 3 contained the exact token `ERR_404`, giving them high lexical scores.
2. **Dense Semantic Complement:** The dense retriever favored Document 2 ("Guide to troubleshooting...") for semantic troubleshooting concepts.
3. **RRF Merge:** Document 0 contains BOTH the exact error code (`ERR_404`) and the semantic database context, achieving Rank 1 in the fused hybrid output.

---

### 4. Industry Applications & Trade-offs
- **Elasticsearch / Azure AI Search:** Hybrid search is standard architecture in enterprise search engines.
- **RRF Advantage:** Requires zero manual tuning of score scaling weights between lexical and semantic models.

---

### 5. Top Interview Questions & Answers
**Q1: Why is RRF superior to linear weighted score combination ($alpha \cdot \text{BM25} + (1-\alpha) \cdot \text{Dense}$)?**
> **Answer:** BM25 produces unbounded scores ($0$ to $+\infty$) varying wildly by query length and document length, while cosine similarity ranges in $[-1, 1]$. Normalizing both across disparate queries is notoriously brittle. RRF operates purely on ordinal ranks, making it robust and invariant to score scale distributions.

**Q2: What is the purpose of the constant $k=60$ in RRF?**
> **Answer:** $k$ serves as a damping factor. If $k=0$, Rank 1 scores $1.0$ and Rank 2 scores $0.5$ (a 50% drop). With $k=60$, Rank 1 scores $1/61 \approx 0.0164$ and Rank 2 scores $1/62 \approx 0.0161$, ensuring a document that ranks moderately well in both systems outperforms a document that ranks 1st in one but last in another.